# 06 — Resumable VCOD queue runner
The notebook declares every smoke, tuning, or final cell, inventories stable Drive artifacts, and runs one selected incomplete item. Use `QUEUE_INDEX = -1` to continue with the next incomplete item after a disconnect. Tuning evaluates validation only; final runs require an explicit frozen learning-rate map and protocol approval.

In [ ]:
#@title Select a resumable queue
RUN_KIND = 'exploratory' #@param ['smoke', 'exploratory', 'tuning', 'final']
QUEUE_INDEX = -1 #@param {type:'integer'}
# -1 selects the first incomplete item; otherwise select the displayed zero-based index.
FINAL_MAX_STEPS = 10000 #@param {type:'integer'}
EXPLORATORY_STEPS = 75 #@param {type:'integer'}
EXPLORATORY_TRAIN_TARGETS = 512 #@param {type:'integer'}
EXPLORATORY_VAL_TARGETS = 256 #@param {type:'integer'}
EXPLORATORY_SUBSET_SEED = 42
TUNING_STAGES = (250, 1000, 3000)
TUNING_PROMOTE_COUNTS = (2, 1, 1)
DECLARED_SEEDS = (42, 43, 44)
LEARNING_RATE_GRID = (0.0001, 0.0003, 0.001)
FINAL_PROTOCOL_APPROVED = False #@param {type:'boolean'}
PROJECT_REPO_URL = 'https://github.com/papanag/cod-ssl.git'
PROJECT_BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/cod-ssl'
MAMBA_SSM_VERSION = '2.3.2.post1'  # Keep DT environments reproducible.

In [ ]:
# Fresh-kernel bootstrap using the same cached assets as notebooks 01–05.
from google.colab import drive
drive.mount('/content/drive')
from getpass import getpass
from pathlib import Path
import json, os, subprocess, sys, torch, yaml
project_dir = Path('/content/cod-ssl')
if (project_dir / '.git').is_dir():
    subprocess.run(['git', '-C', str(project_dir), 'fetch', 'origin', PROJECT_BRANCH], check=True)
    subprocess.run(['git', '-C', str(project_dir), 'checkout', PROJECT_BRANCH], check=True)
    subprocess.run(['git', '-C', str(project_dir), 'pull', '--ff-only', 'origin', PROJECT_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', PROJECT_BRANCH, PROJECT_REPO_URL, str(project_dir)], check=True)
# Install ordinary dependencies first. mamba-ssm's build metadata imports torch, so it
# must be installed separately without pip build isolation after Colab's CUDA torch.
pip = [sys.executable, '-m', 'pip']
subprocess.run(pip + ['install', 'setuptools<82', 'wheel', 'packaging', 'ninja'], check=True)
subprocess.run(pip + ['install', '-e', f'{project_dir}[dev,notebooks]'], check=True)
try:
    from importlib.metadata import version
    from mamba_ssm import Mamba
    mamba_ready = version('mamba-ssm') == MAMBA_SSM_VERSION
except Exception:
    mamba_ready = False
if not mamba_ready:
    subprocess.run(pip + ['install', '--no-build-isolation',
                          f'mamba-ssm=={MAMBA_SSM_VERSION}'], check=True)
    try:
        from mamba_ssm import Mamba
    except Exception as error:
        raise RuntimeError(
            'mamba-ssm installed but cannot load with this Colab Torch/CUDA runtime.'
        ) from error
else:
    print('Using installed mamba-ssm', MAMBA_SSM_VERSION)
bootstrap_env = os.environ.copy()
dino_weights = Path(DRIVE_ROOT) / 'checkpoints/dinov3_vitb16.pth'
if not dino_weights.is_file():
    private_url = getpass('Private DINOv3 ViT-B/16 LVD-1689M URL: ').strip()
    if not private_url: raise ValueError('The approved DINOv3 URL is required.')
    bootstrap_env['COD_SSL_DINOV3_DOWNLOAD_URL'] = private_url
    del private_url
state_file = Path('/content/cod_ssl_bootstrap_state.json')
subprocess.run([sys.executable, str(project_dir / 'scripts/bootstrap_colab.py'),
                '--project-dir', str(project_dir), '--drive-root', DRIVE_ROOT,
                '--state-file', str(state_file)],
               cwd=project_dir, env=bootstrap_env, check=True)
bootstrap_env.pop('COD_SSL_DINOV3_DOWNLOAD_URL', None)
state = json.loads(state_file.read_text())
os.environ.update(state['environment'])
PROJECT_DIR = Path(state['project_dir'])
project_src = str(PROJECT_DIR / 'src')
if project_src not in sys.path: sys.path.insert(0, project_src)
VCOD_ROOT = Path(state['drive_root']) / 'vcod'
MOCA_MANIFEST = VCOD_ROOT / 'data/processed/moca_mask_dense_v1/manifest/runtime_manifest.csv'
CAMOTION_MANIFEST = VCOD_ROOT / 'manifests/camotion.csv'
APPROVAL_PATH = VCOD_ROOT / 'approvals/vcod_validation_approval.json'
os.environ['MOCA_MASK_DENSE_MANIFEST'] = str(MOCA_MANIFEST)
os.environ['CAMOTION_MANIFEST'] = str(CAMOTION_MANIFEST)
os.chdir(PROJECT_DIR)
print('Ready on', state['gpu'])

In [ ]:
# Build the current successive-halving stage and select one incomplete item.
import pandas as pd
from IPython.display import display
PRIMARY_CELLS = [(d, r, s) for d, r in [('moca_mask_dense', 'D1'),
                 ('camotion', 'S5')] for s in ('DS', 'VI', 'DT', 'VV')]
ABLATION_CELLS = [('moca_mask_dense', 'S5', s) for s in ('DT', 'VV')]
adapter_by_system = {'DS': 'single', 'VI': 'single',
                     'DT': 'gated_mamba_mix__T64_S1_target32',
                     'VV': 'vjepa_native__T64_S1_target32'}
backbone_by_system = {'DS': 'dinov3_vitb16', 'DT': 'dinov3_vitb16',
                      'VI': 'vjepa21_vitb16', 'VV': 'vjepa21_vitb16'}
def cell_key(dataset, regime, system): return f'{dataset}/{regime}/{system}'
def queue_run_dir(item):
    run_id = (f"{item['dataset']}_{item['regime']}__{item['system']}__"
              f"{backbone_by_system[item['system']]}__{adapter_by_system[item['system']]}"
              f"__seed{item['seed']}")
    if RUN_KIND == 'final': return VCOD_ROOT / 'runs' / run_id
    if RUN_KIND == 'exploratory':
        identity = (f'{run_id}__steps{EXPLORATORY_STEPS}'
                    f'__train{EXPLORATORY_TRAIN_TARGETS}__val{EXPLORATORY_VAL_TARGETS}'
                    f'__balanced_seed{EXPLORATORY_SUBSET_SEED}')
        return VCOD_ROOT / 'exploratory' / identity
    if RUN_KIND == 'tuning':
        return VCOD_ROOT / 'tuning' / f"{run_id}__lr{item['learning_rate']:g}"
    return VCOD_ROOT / 'smoke' / run_id
def stage_dir(item):
    path = queue_run_dir(item)
    return path if RUN_KIND != 'tuning' else path / 'stages' / f"step_{item['target_steps']:06d}"
def queue_status(item):
    path, stage = queue_run_dir(item), stage_dir(item)
    if (stage / 'EVALUATION_COMPLETE').is_file(): return 'complete'
    if (stage / 'TRAINING_COMPLETE').is_file():
        return 'complete' if RUN_KIND == 'smoke' else 'trained'
    if (path / 'checkpoints/last.pt').is_file(): return 'partial'
    return 'pending'
def validation_score(item):
    summary = json.loads((stage_dir(item) / 'summary.json').read_text())
    if summary['run'].get('training_step') != item['target_steps']:
        raise ValueError(f"Wrong checkpoint step in {stage_dir(item)}")
    return float(summary['metrics']['minmax']['video_weighted_study_primary']['S'])
selection_path = VCOD_ROOT / 'approvals/vcod_tuning_selection.json'
if RUN_KIND == 'smoke':
    queue = [dict(dataset=d, regime=r, system=s, seed=42, learning_rate=0.0003,
                  target_steps=1) for d, r, s in PRIMARY_CELLS + ABLATION_CELLS]
elif RUN_KIND == 'exploratory':
    if min(EXPLORATORY_STEPS, EXPLORATORY_TRAIN_TARGETS, EXPLORATORY_VAL_TARGETS) < 1:
        raise ValueError('Exploratory step and target limits must be positive.')
    queue = [dict(dataset='moca_mask_dense', regime='D1', system=s, seed=42,
                  learning_rate=0.0003, target_steps=EXPLORATORY_STEPS)
             for s in ('DS', 'VI', 'DT', 'VV')]
elif RUN_KIND == 'tuning':
    candidates = {cell_key(d, r, s): list(LEARNING_RATE_GRID) for d, r, s in PRIMARY_CELLS}
    queue = None
    for stage_index, target_steps in enumerate(TUNING_STAGES):
        stage_queue = [dict(dataset=d, regime=r, system=s, seed=42, learning_rate=lr,
                            target_steps=target_steps, tuning_stage=stage_index)
                       for d, r, s in PRIMARY_CELLS
                       for lr in candidates[cell_key(d, r, s)]]
        if any(queue_status(item) != 'complete' for item in stage_queue):
            queue = stage_queue
            break
        promoted, rankings = {}, {}
        for d, r, s in PRIMARY_CELLS:
            key = cell_key(d, r, s)
            items = [item for item in stage_queue if cell_key(d, r, s) == key]
            ranked = sorted(((validation_score(item), item['learning_rate']) for item in items),
                            key=lambda pair: (-pair[0], pair[1]))
            rankings[key] = [{'learning_rate': lr, 'val_video_weighted_S': score}
                             for score, lr in ranked]
            promoted[key] = [lr for _, lr in ranked[:TUNING_PROMOTE_COUNTS[stage_index]]]
        receipt_path = VCOD_ROOT / 'tuning/selections' / f'after_step_{target_steps:06d}.json'
        receipt_path.parent.mkdir(parents=True, exist_ok=True)
        receipt = {'schema_version': 1, 'target_steps': target_steps,
                   'selection_metric': 'val_video_weighted_S', 'rankings': rankings,
                   'promoted_learning_rates': promoted}
        if receipt_path.is_file() and json.loads(receipt_path.read_text()) != receipt:
            raise ValueError(f'Existing promotion receipt differs: {receipt_path}')
        if not receipt_path.is_file():
            temporary = receipt_path.with_suffix('.json.part')
            temporary.write_text(json.dumps(receipt, indent=2, sort_keys=True) + '\n')
            temporary.replace(receipt_path)
        candidates = promoted
    if queue is None:
        selected = {key: values[0] for key, values in candidates.items()}
        payload = {'schema_version': 2, 'selection_metric': 'val_video_weighted_S',
                   'tuning_stages': list(TUNING_STAGES),
                   'selected_learning_rates': selected}
        if selection_path.is_file() and json.loads(selection_path.read_text()) != payload:
            raise ValueError(f'Frozen tuning receipt differs: {selection_path}')
        if not selection_path.is_file():
            temporary = selection_path.with_suffix('.json.part')
            temporary.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n')
            temporary.replace(selection_path)
        raise RuntimeError(f'Tuning queue complete; review {selection_path} and select final.')
else:
    if not FINAL_PROTOCOL_APPROVED or not selection_path.is_file():
        raise PermissionError('Complete tuning, review its receipt, and approve final protocol first.')
    selected = json.loads(selection_path.read_text())['selected_learning_rates']
    def selected_lr(d, r, s):
        return float(selected[cell_key(d, 'D1' if d == 'moca_mask_dense' and r == 'S5' else r, s)])
    queue = [dict(dataset=d, regime=r, system=s, seed=seed,
                  learning_rate=selected_lr(d, r, s), target_steps=FINAL_MAX_STEPS)
             for seed in DECLARED_SEEDS for d, r, s in PRIMARY_CELLS + ABLATION_CELLS]
inventory = [{'index': index, **item, 'status': queue_status(item),
              'run_dir': str(queue_run_dir(item))} for index, item in enumerate(queue)]
display(pd.DataFrame(inventory)[['index', 'dataset', 'regime', 'system', 'seed',
                                 'learning_rate', 'target_steps', 'status']])
incomplete = [row['index'] for row in inventory if row['status'] != 'complete']
if not incomplete: raise RuntimeError(f'{RUN_KIND} queue is complete.')
selected_index = incomplete[0] if QUEUE_INDEX == -1 else QUEUE_INDEX
if selected_index not in range(len(queue)): raise IndexError(f'QUEUE_INDEX must be -1 or 0..{len(queue)-1}')
if inventory[selected_index]['status'] == 'complete':
    raise ValueError(f'Queue item {selected_index} is complete; use -1 for next incomplete.')
active = queue[selected_index]
DATASET, REGIME, SYSTEM = active['dataset'], active['regime'], active['system']
SEED, LEARNING_RATE, TARGET_STEPS = active['seed'], active['learning_rate'], active['target_steps']
RUN_DIR, STAGE_DIR = queue_run_dir(active), stage_dir(active)
print('Selected queue item:', selected_index, active, 'status=', queue_status(active))

In [ ]:
# Validate the selected queue item and the signed dataset/checkpoint gate.
from cod_ssl.utils.run import file_sha256
manifest = MOCA_MANIFEST if DATASET == 'moca_mask_dense' else CAMOTION_MANIFEST
if not manifest.is_file(): raise FileNotFoundError(manifest)
if not APPROVAL_PATH.is_file():
    raise PermissionError('Run notebook 05 and complete its manual sign-off first.')
approval = json.loads(APPROVAL_PATH.read_text())
expected_hash = approval['moca_manifest_sha256' if DATASET == 'moca_mask_dense' else 'camotion_manifest_sha256']
if file_sha256(manifest) != expected_hash:
    raise ValueError('Manifest changed after manual approval; rerun notebook 05.')
if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
    raise RuntimeError('The locked run requires a BF16-capable Colab GPU.')
RUN_DIR.mkdir(parents=True, exist_ok=True)
properties = torch.cuda.get_device_properties(0)
print({'queue_index': selected_index, 'gpu': properties.name,
       'memory_gib': round(properties.total_memory / 2**30, 2), **active,
       'kind': RUN_KIND, 'run_dir': str(RUN_DIR)})

In [ ]:
# Train or resume. Periodic atomic checkpoints are written directly to the stable Drive run.
config_path = ('configs/experiments/vcod_diagnostics.yaml' if SYSTEM in {'DM', 'VR'}
               else 'configs/experiments/vcod_primary_2x2.yaml')
manifest_env = 'MOCA_MASK_DENSE_MANIFEST' if DATASET == 'moca_mask_dense' else 'CAMOTION_MANIFEST'
source_stride = 5 if REGIME == 'S5' else 1
released_stride = 1 if DATASET == 'camotion' else source_stride
release_profile = 'moca_mask_dense_v1' if DATASET == 'moca_mask_dense' else 'camotion_public_stride5_v1'
boundary_policy = 'manual_target_hull_v1' if DATASET == 'moca_mask_dense' else 'public_sequence_extent_v1'
context_cadence = 'dense_source_stride1' if source_stride == 1 else 'source_stride5'
command = [sys.executable, 'scripts/train_probe.py', '--config', config_path,
           '--run-dir', str(RUN_DIR),
           f'experiment.system_id={SYSTEM}', f'experiment.seed={SEED}',
           f'dataset.name={DATASET}', f'dataset.regime={REGIME}',
           f'dataset.manifest_env={manifest_env}',
           f'dataset.release_profile={release_profile}', f'dataset.boundary_policy={boundary_policy}',
           f'dataset.dense_intermediate_rgb_available={str(DATASET == "moca_mask_dense").lower()}',
           f'clip.stride={released_stride}', f'clip.released_stride={released_stride}',
           f'clip.source_frame_stride={source_stride}', f'clip.context_cadence={context_cadence}',
           f'training.learning_rate={LEARNING_RATE}', f'training.max_steps={TARGET_STEPS}',
           f'evaluation.save_logits={str(RUN_KIND == "final").lower()}']
if RUN_KIND == 'exploratory':
    command += ['experiment.primary=false', 'experiment.exploratory=true',
                f'training.limit_targets={EXPLORATORY_TRAIN_TARGETS}',
                f'training.subset_seed={EXPLORATORY_SUBSET_SEED}',
                f'evaluation.limit_targets={EXPLORATORY_VAL_TARGETS}',
                f'evaluation.subset_seed={EXPLORATORY_SUBSET_SEED}']
training_complete = STAGE_DIR / 'TRAINING_COMPLETE'
checkpoint = RUN_DIR / 'checkpoints/last.pt'
if training_complete.is_file():
    print('Training already complete; preserving it and continuing to evaluation.')
else:
    if checkpoint.is_file():
        command += ['--resume', str(checkpoint)]
        print('Resuming partial training from', checkpoint)
    if RUN_KIND == 'smoke': command.append('--smoke')
    subprocess.run(command, cwd=PROJECT_DIR, check=True)
if hasattr(os, 'sync'): os.sync()

In [ ]:
# Evaluate each tuning stage separately; final evaluation remains at the run root.
evaluation_complete = STAGE_DIR / 'EVALUATION_COMPLETE'
if RUN_KIND != 'smoke' and not evaluation_complete.is_file():
    split = 'val' if RUN_KIND in {'exploratory', 'tuning'} else 'test'
    evaluate_command = [sys.executable, 'scripts/evaluate.py', '--run-dir', str(RUN_DIR),
                        '--checkpoint', str(RUN_DIR / 'checkpoints/last.pt'), '--split', split]
    if RUN_KIND == 'tuning': evaluate_command += ['--output-dir', str(STAGE_DIR)]
    if RUN_KIND == 'final': evaluate_command.append('--save-logits')
    subprocess.run(evaluate_command, cwd=PROJECT_DIR, check=True)
    if hasattr(os, 'sync'): os.sync()
elif evaluation_complete.is_file():
    print('Evaluation already complete; preserving existing artifacts.')
else:
    print('Smoke evaluation intentionally skipped.')

In [ ]:
# Verify the selected item, then show the remaining queue.
from IPython.display import JSON, display
required = [RUN_DIR / 'config_resolved.yaml', RUN_DIR / 'environment.json',
            RUN_DIR / 'split_ids.json', RUN_DIR / 'train_log.jsonl',
            RUN_DIR / 'checkpoints.json', RUN_DIR / 'checkpoints/last.pt']
required.append(STAGE_DIR / 'TRAINING_COMPLETE')
if RUN_KIND != 'smoke': required.append(STAGE_DIR / 'EVALUATION_COMPLETE')
if RUN_KIND == 'exploratory':
    required += [RUN_DIR / 'training_subset.json', STAGE_DIR / 'evaluation_subset.json']
missing = [str(path) for path in required if not path.is_file()]
if missing: raise FileNotFoundError(f'Missing run artifacts: {missing}')
log = pd.read_json(RUN_DIR / 'train_log.jsonl', lines=True)
display(log.tail(20))
if (STAGE_DIR / 'summary.json').is_file():
    display(JSON(json.loads((STAGE_DIR / 'summary.json').read_text())))
print('Completed queue item:', selected_index, RUN_DIR)
for row, item in zip(inventory, queue):
    row['status'] = queue_status(item)
remaining = [row for row in inventory if row['status'] != 'complete']
display(pd.DataFrame(remaining)[['index', 'dataset', 'regime', 'system', 'seed',
                                 'learning_rate', 'target_steps', 'status']]) if remaining else print(
    'Current stage complete; rerun from the queue cell to promote candidates.')